# Code Causal LM

源码导航：[`WalkieConfig`](../../../core/model/walkie.py#L39)、[`WalkieBlock`](../../../core/model/walkie.py#L79)、[`WalkieForCausalLM`](../../../core/model/walkie.py#L114)。

模型主体把模块组合成 decoder-only causal LM：Embedding → 多层 pre-norm block → final RMSNorm → LM head。每个 block 的残差结构是：

$$
x_{l+1/2}=x_l+\operatorname{Attn}(\operatorname{RMSNorm}(x_l))
$$

$$
x_{l+1}=x_{l+1/2}+\operatorname{SwiGLU}(\operatorname{RMSNorm}(x_{l+1/2}))
$$

这里的改进思路是把 GPT-2 baseline 的 LayerNorm + MHA + GELU MLP 替换为 RMSNorm + GQA/QK-Norm/RoPE + SwiGLU，并用 tied embeddings 把 65,536 词表控制在 1B 参数预算内。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.model.walkie import WalkieConfig, WalkieForCausalLM

## 1. Tiny 前向 / loss / generate

In [ ]:
cfg = WalkieConfig(
    vocab_size=128, block_size=64, n_embd=64, n_layer=2,
    n_head=4, n_head_kv=2, head_dim=16, d_ffn=128,
    dropout=0.0, bias=False, tie_weights=True,
)
model = WalkieForCausalLM(cfg)
idx = torch.randint(0, cfg.vocab_size, (2, 8))
targets = torch.randint(0, cfg.vocab_size, (2, 8))
logits, loss = model(idx, targets)
print('logits:', tuple(logits.shape))
print('loss:', float(loss))
print('params:', model.num_parameters())
print('tied:', model.lm_head.weight.data_ptr() == model.tok_embeddings.weight.data_ptr())
print('generated:', tuple(model.generate(idx[:1], max_new_tokens=4, temperature=0.0).shape))

## 2. 1B 配置解析估算

In [ ]:
cfg = WalkieConfig()
V, D, L = cfg.vocab_size, cfg.n_embd, cfg.n_layer
H, Hkv, Hd, F = cfg.n_head, cfg.n_head_kv, cfg.head_dim, cfg.d_ffn
embed = V * D
per_layer = D * H * Hd + 2 * D * Hkv * Hd + H * Hd * D + 3 * D * F + 2 * D + 2 * Hd
total = embed + L * per_layer + D
print(f'estimated params = {total/1e6:.2f}M')
print('under 1B:', total < 1_000_000_000)

---

## 延伸阅读与参考资料

### 核心论文
- **GPT-2**: Radford et al., 2019. [report](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)
- **Code Llama**: Roziere et al., 2023. [arXiv:2308.12950](https://arxiv.org/abs/2308.12950)
- **DeepSeek-Coder**: Guo et al., 2024. [arXiv:2401.14196](https://arxiv.org/abs/2401.14196)

### 工程实现
- **nanoGPT**: [GitHub](https://github.com/karpathy/nanoGPT)
- **Hugging Face Transformers causal LM implementations**: [source](https://github.com/huggingface/transformers/tree/main/src/transformers/models)